In [1]:
# import libraries
import polars as pl
from datetime import date
from tqdm.notebook import tqdm
import plotly.express as px

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_width_chars(200)

# data path
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from src.config.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR, DOCS_DIR

In [2]:
# load cleaned data
df = pl.scan_parquet(INTERIM_DIR / "cleaned.parquet")

# 8. Data Model Preparation

**Main Objective**

Transform the selected origination-time features into a model-ready representation while preserving predictive information, interpretability, and leakage control.

## 8.1 One Hot Encoding

In [3]:
CAT_FEATURES = [i for i,d in df.collect_schema().items() if not d.is_numeric()]
CAT_FEATURES.remove("issue_d")

CAT_FEATURE_VALUES = {}

for feature in CAT_FEATURES:
    values = (
        df
        .select(
            pl.col(feature)
            .drop_nulls()
            .cast(pl.String)
            .unique()
            .sort()
        )
        .collect()
        .to_series()
        .to_list()
    )

    CAT_FEATURE_VALUES[feature] = values

In [4]:
print(CAT_FEATURE_VALUES)

{'grade': ['A', 'B', 'C', 'D', 'E', 'F', 'G'], 'initial_list_status': ['f', 'w'], 'application_type': ['Individual', 'Joint App'], 'purpose': ['car', 'credit_card', 'debt_consolidation', 'educational', 'home_improvement', 'house', 'major_purchase', 'medical', 'moving', 'other', 'renewable_energy', 'small_business', 'vacation', 'wedding'], 'home_ownership': ['ANY', 'MORTGAGE', 'NONE', 'OWN', 'RENT'], 'verification_status': ['Not Verified', 'Source Verified', 'Verified'], 'addr_state': ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']}


In [5]:
from src.config.categories import OHE_CATEGORIES

In [8]:
def one_hot_encode(df: pl.DataFrame | pl.LazyFrame,):
    expressions = []

    for feature, categories in OHE_CATEGORIES.items():

        for category in categories:

            encoded_name = f"{feature}_{category}"

            expressions.append(
                (
                    pl.col(feature).cast(pl.String)
                    == pl.lit(category)
                )
                .fill_null(False)
                .cast(pl.UInt8)
                .alias(encoded_name)
            )

    return (df
            .with_columns(expressions)
            .select(pl.exclude(OHE_CATEGORIES.keys())))

In [9]:
df = one_hot_encode(df)

In [11]:
df.head().sort("issue_d").collect()

loan_amnt,term,int_rate,emp_length,annual_inc,fico_range_low,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,num_tl_op_past_12m,dti,total_acc,open_act_il,open_il_12m,open_rv_24m,all_util,total_cu_tl,pct_tl_nvr_dlq,mort_acc,tot_cur_bal,num_actv_rev_tl,revol_util,max_bal_bc,mths_since_recent_bc,total_bc_limit,mths_since_rcnt_il,total_bal_il,il_util,inq_last_6mths,inq_fi,inq_last_12m,delinq_2yrs,num_accts_ever_120_pd,num_tl_120dpd_2m,num_tl_90g_dpd_24m,collections_12_mths_ex_med,pub_rec,pub_rec_bankruptcies,tot_coll_amt,num_il_tl,mths_since_recent_inq_null,mths_since_last_record_null,mths_since_last_major_derog_null,emp_length_null,cr_age_mths,default,issue_d,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,initial_list_status_f,initial_list_status_w,application_type_Individual,application_type_Joint App,purpose_car,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,home_ownership_ANY,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OWN,home_ownership_RENT,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified,addr_state_AK,addr_state_AL,addr_state_AR,addr_state_AZ,addr_state_CA,addr_state_CO,addr_state_CT,addr_state_DC,addr_state_DE,addr_state_FL,addr_state_GA,addr_state_HI,addr_state_ID,addr_state_IL,addr_state_IN,addr_state_KS,addr_state_KY,addr_state_LA,addr_state_MA,addr_state_MD,addr_state_ME,addr_state_MI,addr_state_MN,addr_state_MO,addr_state_MS,addr_state_MT,addr_state_NC,addr_state_ND,addr_state_NE,addr_state_NH,addr_state_NJ,addr_state_NM,addr_state_NV,addr_state_NY,addr_state_OH,addr_state_OK,addr_state_OR,addr_state_PA,addr_state_RI,addr_state_SC,addr_state_SD,addr_state_TN,addr_state_TX,addr_state_UT,addr_state_VA,addr_state_VT,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY
i64,i64,f64,i64,f64,i64,f64,i64,i64,i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,i8,i8,i8,i8,i64,i8,date,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
12000,36,7.97,0,42000.0,715,131.0,255,1,1,3,27.74,16,2.0,1.0,4.0,53.0,1.0,100.0,0,30502.0,6,37.0,7117.0,14,15500.0,8,19045.0,73.0,0.0,1.0,2.0,0,0,0.0,0,0,1,1,0.0,7,0,0,1,1,258,0,2017-09-01,1,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10000,36,9.44,3,55000.0,695,144.0,73,49,10,1,18.79,10,4.0,1.0,0.0,68.0,1.0,100.0,2,340607.0,2,57.5,6847.0,49,10500.0,10,209187.0,75.0,0.0,1.0,1.0,0,0,0.0,0,0,1,1,0.0,5,0,0,1,0,146,0,2017-09-01,0,1,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8000,36,16.02,0,120000.0,700,137.0,276,34,6,3,20.36,34,3.0,2.0,0.0,88.0,24.0,97.1,6,388595.0,3,92.3,0.0,152,0.0,6,83572.0,85.0,1.0,1.0,2.0,0,0,0.0,0,0,0,0,0.0,19,0,1,1,0,280,0,2017-09-01,0,0,1,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
12800,36,13.59,5,90000.0,660,154.0,345,5,5,3,22.63,23,3.0,2.0,3.0,86.0,1.0,83.0,0,93375.0,6,86.0,3777.0,5,14750.0,6,80715.0,91.0,2.0,0.0,2.0,0,0,0.0,0,0,0,0,0.0,15,0,1,0,0,350,0,2017-09-01,0,0,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
15000,36,13.59,4,180000.0,680,132.0,489,16,13,0,38.07,50,4.0,0.0,1.0,68.0,8.0,100.0,4,682000.0,14,66.6,22127.0,16,116700.0,13

## 8.2 Temporal Split

In [28]:
total_rows = df.select(pl.len()).collect().item()

issue_d_cumsum = (df
                    .select("issue_d")
                    .group_by("issue_d")
                    .agg(pl.len().alias("count"))
                    .sort("issue_d")
                    .with_columns(
                        (pl.col("count") / total_rows * 100.0).cast(pl.Float64).round(2).alias("pct"),
                        (pl.col("count").cum_sum().alias("count_cumsum"))
                    )
                    .with_columns(
                        (pl.col("count_cumsum") / total_rows * 100.0).cast(pl.Float64).round(2).alias("pct_cumsum")
                    )
                    .collect())

In [34]:
import plotly.express as px
import pandas as pd

df_pd = issue_d_cumsum.to_pandas()

# Convert the first and last dates to strings with format YYYY-MM-DD
first_date_str = df_pd['issue_d'].min().strftime('%Y-%m-%d')
last_date_str = df_pd['issue_d'].max().strftime('%Y-%m-%d')

fig = px.line(
    df_pd,
    x="issue_d",
    y="pct_cumsum",
    markers=True,
    title="Data Cumulative Split (85 / 10 / 5)",
    labels={
        "issue_d": "Vintage",
        "pct_cumsum": "Percentage",
    },
)

# 1. Highlight Train Area (Start to Jul 2018)
fig.add_vrect(
    x0=first_date_str, 
    x1="2018-07-01",
    fillcolor="green", 
    opacity=0.1, 
    layer="below", 
    line_width=0,
    annotation_text="Train (85%)", 
    annotation_position="top left"
)

# 2. Highlight Validation Area (Aug 2018 to Mar 2019)
fig.add_vrect(
    x0="2018-07-01", 
    x1="2019-03-01",
    fillcolor="orange", 
    opacity=0.1, 
    layer="below", 
    line_width=0,
    annotation_text="Val (10%)", 
    annotation_position="top left"
)

# 3. Highlight Test Area (Apr 2019 to End)
fig.add_vrect(
    x0="2019-03-01", 
    x1=last_date_str,
    fillcolor="red", 
    opacity=0.1, 
    layer="below", 
    line_width=0,
    annotation_text="Test (5%)", 
    annotation_position="top left"
)

fig.show()

In [ ]:
df_train = df.filter(pl.col("issue_d")<pl.Date(20))